<a href="https://colab.research.google.com/github/pia-francesca/ema/blob/main/examples/Pla2g2/emmaemb_pla2g2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EmmaEmb: Comparative Analysis of Embedding Spaces

Welcome to the example Colab notebook for EmmaEmb, a Python library for analyzing and comparing embedding spaces in molecular biology. EmmaEmb provides tools to explore how different embedding models capture biological information, enabling insights into feature similarities, differences, and relationships across embeddings.

Link to GitHub: https://github.com/broadinstitute/EmmaEmb


### Notebook content

This notebook demonstrates key functionalities of EmmaEmb, including:

I. [Initialising and populating the Emma object](#section-one) (Figure 1A-B)

II. [Embedding space diagnostics](#section-two)

III. [Feature distribution across spaces](#section-three) (Figure 1C)

IV. [Pairwise space comparison](#section-four) (Figure 1D)

### Diagnostic workflow for embedding space ranking

Included in this workflow is a **diagnostic workflow for embedding space ranking**: a 6-step workflow for evaluating and comparing the quality of different embedding spaces for encoding a specific feature. Steps 1-2 characterize the geometric properties of the embedding space (section II.) and are run once per embedding space prior to any feature-specific analysis. Steps 3-6 (section III.) are run separately for each feature of interest, as class distribution, k range, and robustness assessments depend on the specific feature being analyzed.


| Step | Purpose |
|---|---|
| **1. Anisotropy assessment** | Detect directional bias; apply mean-centering where needed and select the appropriate distance metric |
| **2. Hubness assessment** | Identify whether a small number of hub points dominate k-nearest-neighbour lists |
| **3. Class distribution and k range** | Count samples per class, derive n_min as a hard upper bound on k, compute random baselines |
| **4. Parameter sensitivity analysis** | Measure KNN alignment across the full k range and all distance metrics |
| **5. Class imbalance robustness** | Progressively downsample the majority class to verify imbalance is not driving rankings |
| **6. Feature noise robustness** | Progressively perturb feature labels to quantify how tight class structure is |

> Disclaimer: The decision thresholds used throughout this workflow (e.g. RHI > 0.3 for high hubness, average cosine similarity > 0.3 for anisotropy, noise crossing point > 0.3 for robustness) are chosen based on empirical observation rather than established literature benchmarks. They are intended as a reasonable starting point for interpretation. Further empirical evaluation on a broader range of datasets and tasks will be needed to validate or refine these thresholds.


![EmmaEmb Overview](https://raw.githubusercontent.com/broadinstitute/EmmaEmb/main/images/emma_overview.jpg)
Figure 1: **Schematic of the EmmaEmb framework and proposed workflow.** (A) Starting with a set of data samples (e.g., genes or proteins, here: s1-s6, embeddings are derived from different embedding models (here: Embedding Models, V1-V3). For protein biology applications, embeddings can be derived from protein language models that vary in their architecture or training. (B) Categorical features (here: f1-f3) associated with the samples are integrated into the analysis. (C) Analysis approach 1: Feature distribution across spaces focuses on examining how samples with specific features, such as protein families, are grouped within the neighborhoods within an embedding space as well as across spaces. Multiple k-nearest neighbor (KNN)-based methods can be applied within this approach to quantify the proximity of samples with the same feature as well as the proximity of groups of samples with the same feature. (D) Analysis approach 2: Pairwise space comparison involves the identification of similarities and differences in distances between embeddings or KNNs between two embedding spaces. Regions of notable difference can be further analyzed using the features to characterize samples that are represented differently across the models. Methods in (C) and (D) are modular and can be applied in any order, either independently within each approach or in combination between the approaches, depending on the analysis goals.


## Loading dependencies and data

Information of the data and embedding models can be found here: https://github.com/broadinstitute/EmmaEmb/tree/main/examples/Pla2g2/README.md

In [ ]:
#@title Install dependencies
%pip install --upgrade emmaemb

In [ ]:
import os
import math
import requests
import pandas as pd
import numpy as np
import plotly.express as px

import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [ ]:
#@title Download example data from EmmaEmb repository

# download embeddings

models = ["ESMC", "ProtT5"]
embedding_url_dir = "https://raw.githubusercontent.com/broadinstitute/EmmaEmb/main/examples/Pla2g2/embeddings/"

headers = {"User-Agent": "Mozilla/5.0"}
csv_url = "https://raw.githubusercontent.com/broadinstitute/EmmaEmb/main/examples/Pla2g2/Pla2g2_features.csv"

csv_filename = "Pla2g2_features.csv"
csv_response = requests.get(csv_url, headers=headers)
if csv_response.status_code == 200:
    with open(csv_filename, "wb") as f:
        f.write(csv_response.content)
else:
    print(f"Failed to download {csv_filename}")


df_pla2g2 = pd.read_csv(csv_filename)
proteins = df_pla2g2['identifier'].values

# now for each model download embedding files for each protein
for model in models:
  model_dir = f"embeddings/{model}"
  os.makedirs(model_dir, exist_ok=True)

  for protein in proteins:
    file_path = os.path.join(model_dir, f"{protein}.npy")

    # Check if file already exists
    if os.path.exists(file_path):
        continue

    url = f"{embedding_url_dir}{model}/{protein}.npy"
    response = requests.get(url, headers=headers)

    if response.status_code == 200:
      # store in embeddings/model/protein-id.npy
      with open("embeddings/" + model + "/" + protein + ".npy", "wb") as f:
        f.write(response.content)
    else:
      print(f"Failed to download {url}")


print("All download of feature data complete.")

<a name="section-one"></a>
## I. Initialising and populating the Emma object

### Initialising the Emma object with the feature data

The first column of the feature data should include the identifiers of the samples, in this case proteins. The remaining columns contain meta data on each sample.

In [ ]:
df_pla2f2 = pd.read_csv("Pla2g2_features.csv")
print(df_pla2f2.shape)
df_pla2f2.head()

Initialising Emma object with the feature data in format of a pandas df. The datatype stored in each column of the feature data is detected. Only cateorical data will be available for downstream analysis with the Emma library.

Note: Quantitative features can be binned to allow analysis with EmmaEmb.

In [ ]:
# initiate Emma object with the metadata
from emmaemb import Emma

emma = Emma(df_pla2f2)

### Adding embedding spaces

Embedding spaces can be added one by one. Either by

- providing a link to a directory which stores the embeddings in individual files with the identifiers from the feature table or

- by providing a numpy array which includes the embeddings in each row and in the same order as in the feature data table.


Multiple embedding spaces can be added. Dimensions of the embeddings do not have to be the same across embedding spaces. Embedding spaces can be removed using the `remove_emb_space(emb_space_name: str)` function.

In [ ]:
embedding_dir = "embeddings/"
models = ["ProtT5", "ESMC"]

In [ ]:
for model_name in models:
    emma.add_emb_space(
        embeddings_source=embedding_dir + model_name,
        emb_space_name=model_name,
    )

### Visualization of embedding spaces using dimensionality reduction techniques

The `plot_emb_space` function visualizes the embeddings of a specified embedding space in 2D using dimensionality reduction techniques such as PCA, t-SNE, or UMAP. It takes an Emma instance containing multiple embedding spaces and projects the selected space into two dimensions. The resulting scatter plot can be colored based on metadata attributes, allowing for an initial visual inspection of the embedding space.

In [ ]:
# visualise reduced embedding space
from emmaemb.visualization import plot_emb_space

fig_pca = plot_emb_space(emma=emma,
                         emb_space="ProtT5",
                         method="PCA",
                         color_by="enzyme_class",
                         normalize=True)
fig_pca.show()

<a name="section-two"></a>
## II. Embedding space diagnostics

This section includes functions to characterize the geometric properties of the embedding space and adjust the parameters and methods for downstream analyses. These functions are run once per embedding space prior to any feature-specific analysis.

### Step 1: Anisotropy assessment

An isotropic embedding space is one where vectors point in many different directions. High anisotropy, where most embeddings cluster along a narrow cone, can inflate cosine similarities and distort nearest-neighbor structure.

`get_anisotropy_diagnostics` measures the average pairwise cosine similarity between random vector pairs. Values close to 0 indicate near-isotropic geometry; values approaching 1 indicate severe anisotropy. The diagnostic also reports whether mean-centering is likely to help.

In [ ]:
from emmaemb.functions import get_anisotropy_diagnostics

feature          = "enzyme_class"
distance_metrics = ["cosine", "cityblock"]

aniso = get_anisotropy_diagnostics(emma, n_pairs=10_000, seed=42)
print(aniso)

Mean-centering subtracts the per-dimension mean, shifting the embedding cloud to the origin. The cell below automatically detects which embedding spaces benefit from centering (those where it resolves the anisotropy score) and applies it in-place. All cached pairwise distances are cleared and recomputed for all metrics.

In [ ]:
anisotropic_spaces = (
    aniso.data["summary"]
    .dropna(subset=["AnisotropyScore_MC"])["Embedding"]
    .tolist()
)
if anisotropic_spaces:
    print(f"Applying mean-centering to: {anisotropic_spaces}")
    emma.mean_center(emb_spaces=anisotropic_spaces)
else:
    print("No mean-centering applied (all spaces are isotropic).")

The `calculate_pairwise_distances` method computes pairwise distances between samples in an embedding space and caches the k-nearest neighbor ranks for downstream analyses. Supported distance metrics include Euclidean, Manhattan, Cosine, and several normalized variants. Distances are only computed once: subsequent calls for the same space and metric return immediately.

In [ ]:
for metric in distance_metrics:
    for model in models:
        emma.calculate_pairwise_distances(model, metric)
        print(f"  {model} / {metric} done")

### Step 2: Hubness assessment

Hubness is a phenomenon in high-dimensional spaces where a small number of points become the nearest neighbors of a disproportionately large number of others, not because they are genuinely similar to everything, but as a mathematical consequence of high dimensionality. Hub points can inflate KNN-based alignment scores and bias downstream analyses.

`get_hubness_diagnostics` returns the Robin Hood index (a summary of how unequally k-occurrences are distributed across samples) and the per-sample k-occurrence distribution.

In [ ]:
from emmaemb.functions import get_hubness_diagnostics

hub = get_hubness_diagnostics(emma, k=10, metric="cosine")
rhi_max_ = hub.data["rhi"]["RobinHoodIndex"].max()
if rhi_max_ > 0.3:
    print("Hubness > 0.3 -> Neighbourhood structure may be distorted.")
print(hub)

In [ ]:
# plot RHI distribution across embedding spaces
fig = px.bar(x=hub.data["rhi"]["Embedding"], y=hub.data["rhi"]["RobinHoodIndex"], title="Robin Hood Index (RHI) distribution per embedding space", labels={"y": "Robin Hood Index", "x": "Embedding spaces"})
fig.update_layout(font=dict(family="Arial", color="black"), template="simple_white", width=500)
fig.show()

<a name="section-three"></a>
## III. Feature distribution across spaces

Tests for the distribution of a specific feature across the neighbourhood of each embedding space. In this example, we will examine the distribution of the enzyme classes across the neighbourhood of each embedding space.

### Step 3: Class distribution and k range


In [ ]:
from emmaemb.visualization import plot_knn_alignment_vs_feature_noise

fig_noise, df_noise = plot_knn_alignment_vs_feature_noise(
    emma=emma,
    feature=feature,
    emb_spaces=models,
    k_values=k_values,
    metrics=distance_metrics,
    n_noise_steps=8,
    n_repeats=3,
    seed=42,
    show_random_baselines=True,
    return_data=True,
)
fig_noise.update_layout(height=500, width=900)
fig_noise.show()

# Step 6: noise robustness
metric_display = {"cosine": "Cosine distance", "cityblock": "Manhattan distance", "euclidean": "Euclidean distance"}
k_ref = k_values[len(k_values) // 2]
primary_metric_label = metric_display.get(distance_metrics[0], distance_metrics[0])

df_n = (
    df_noise[(df_noise["k"] == k_ref) & (df_noise["Metric"] == primary_metric_label)]
    .groupby(["Noise fraction", "Embedding"])["Mean alignment score"]
    .mean()
    .unstack("Embedding")
)

if not df_n.empty and len(df_n.columns) > 1:
    leader_at_zero = df_n.iloc[0].idxmax()
    crossings = df_n[df_n.idxmax(axis=1) != leader_at_zero]
    if crossings.empty or float(crossings.index[0]) > 0.3:
        noise_flag = "ROBUST"
        crossing_frac = float(crossings.index[0]) if not crossings.empty else None
        noise_note = (
            f"Rankings stable up to noise={crossing_frac:.2f}."
            if crossing_frac is not None
            else "Rankings stable across all noise levels."
        )
    else:
        noise_flag = "SENSITIVE"
        crossing_frac = float(crossings.index[0])
        noise_note = f"Rankings flip at noise={crossing_frac:.2f} — class structure is noise-sensitive."
else:
    noise_flag = "ROBUST"
    noise_note = "Single model or no data — noise robustness not applicable."

print(f"\nStep 6 — Noise flag: {noise_flag}")
print(f"  {noise_note}")

# Overall confidence assessment
df_ref = df_knn[df_knn["k"] == k_ref].groupby(["distance_metric", "Embedding"])["Fraction"].mean()
margins = []
for metric_key in distance_metrics:
    if metric_key in df_ref.index.get_level_values("distance_metric"):
        grp = df_ref.xs(metric_key, level="distance_metric").sort_values(ascending=False)
        if len(grp) >= 2:
            margins.append(float(grp.iloc[0] - grp.iloc[1]))
margin = float(np.mean(margins)) if margins else 0.0

all_robust = stability_flag == "STABLE" and balance_flag == "CONSISTENT" and noise_flag == "ROBUST"
any_unreliable = (
    stability_flag == "PARAMETER-SENSITIVE"
    or balance_flag == "INCONSISTENT"
    or noise_flag == "SENSITIVE"
)

if all_robust and margin > 0.10 and rhi_max_ < 0.3:
    confidence = "HIGH"
elif any_unreliable or margin <= 0.05:
    confidence = "LOW"
else:
    confidence = "MODERATE"

print(f"\n=== Final Assessment ===")
print(f"  Stability:    {stability_flag}")
print(f"  Balance:      {balance_flag}")
print(f"  Noise:        {noise_flag}")
print(f"  Hubness:      {'HIGH' if rhi_max_ > 0.3 else 'acceptable'} (RHI_max={rhi_max_:.3f})")
print(f"  Margin:       {margin:.3f}")
print(f"  → Confidence: {confidence}")
print(f"  → Primary model: {primary_model}")

class_counts = emma.metadata[feature].value_counts()
n_min = int(class_counts.min())
n_total = len(emma.metadata)
if n_min < 10 and n_total > 100:
    n_min = 40

k_max_hard = n_min - 1
if rhi_max_ > 0.3:
    k_max = max(2, min(k_max_hard, 20))
else:
    k_max = min(k_max_hard, int(math.sqrt(n_total)))

k_max = max(k_max, 2)
k_values = sorted(set([
    max(2, k_max // 5),
    max(2, k_max // 3),
    max(2, k_max // 2),
    k_max,
]))
k_values = [k for k in k_values if k < n_min]

print(f"\n=== Step 3: Class distribution (feature='{feature}') ===\n")
for cls, cnt in class_counts.items():
    print(f"  {cls}: {cnt} samples")
freq_weighted = sum((cnt / n_total) ** 2 for cnt in class_counts)
print(f"\nn_min={n_min}, N={n_total}")
print(f"Random baselines: uniform=1/{len(class_counts)}={1/len(class_counts):.3f}, "
      f"freq-weighted={freq_weighted:.3f}")
print(f"k range: {k_values}")

fig = px.histogram(emma.metadata[feature], x=feature, title="Class distribution")
fig.update_layout(font=dict(family="Arial", color="black"), template="simple_white")
fig.show()

### Step 4: Parameter sensitivity analysis

The choice of k affects alignment scores. `plot_knn_alignment_across_k` sweeps k across a range and plots the mean alignment score for each embedding space. Stable rankings across k indicate robust results; crossings or inversions suggest parameter sensitivity and should be reported.

In [ ]:
from emmaemb.visualization import plot_knn_alignment_across_k

fig_knn_k, df_knn = plot_knn_alignment_across_k(
    emma=emma,
    feature=feature,
    k_values=k_values,
    metrics=distance_metrics,
    show_random_baselines=True,
    elbow_detection=True,
    return_data=True,
)
fig_knn_k.update_layout(height=500, width=750)
fig_knn_k.show()

# Step 4: parameter sensitivity
top_per_k = (
    df_knn.groupby(["distance_metric", "k"])
    .apply(lambda g: g.loc[g["Fraction"].idxmax(), "Embedding"])
    .rename("top_model")
    .reset_index()
)

top_per_metric = {
    metric: top_per_k[top_per_k["distance_metric"] == metric]["top_model"].mode()[0]
    for metric in distance_metrics
    if metric in top_per_k["distance_metric"].values
}

all_k_stable = len(top_per_k["top_model"].unique()) == 1
metrics_agree = len(set(top_per_metric.values())) == 1

if all_k_stable and metrics_agree:
    stability_flag = "STABLE"
    primary_model = top_per_k["top_model"].mode()[0]
    stability_rec = f"Rankings consistent across k and metrics. Recommend {primary_model}."
elif all_k_stable and not metrics_agree:
    stability_flag = "METRIC-SENSITIVE"
    primary_model = top_per_metric.get("cosine", list(top_per_metric.values())[0])
    stability_rec = f"Stable across k but metric-dependent. Primary (cosine): {primary_model}."
else:
    stability_flag = "PARAMETER-SENSITIVE"
    primary_model = top_per_k["top_model"].mode()[0]
    stability_rec = "Rankings invert across k. Results are parameter-sensitive."

print(f"\nStep 4 | Stability flag: {stability_flag}")
print(f"  {stability_rec}")
print(f"  Top model per metric: {top_per_metric}")

### Step 5: Class imbalance robustness (run if flagged in Step 3)

`plot_knn_alignment_vs_class_balance` progressively downsamples the majority class toward the size of the smallest class and repeats KNN alignment. Stable rankings across this sweep indicate that the result is not driven by class frequency.

In [ ]:
from emmaemb.visualization import plot_knn_alignment_vs_class_balance

fig_balance, df_balance = plot_knn_alignment_vs_class_balance(
    emma=emma,
    feature=feature,
    emb_spaces=models,
    k_values=k_values,
    metrics=distance_metrics,
    n_balance_steps=6,
    seed=42,
    show_random_baselines=True,
    return_data=True,
)
fig_balance.update_layout(height=900, width=750)
fig_balance.show()

# Step 5: class imbalance robustness
metric_display = {"cosine": "Cosine distance", "cityblock": "Manhattan distance", "euclidean": "Euclidean distance"}
k_ref = k_values[len(k_values) // 2]
primary_metric_label = metric_display.get(distance_metrics[0], distance_metrics[0])

df_b = df_balance[
    (df_balance["k"] == k_ref) & (df_balance["Metric"] == primary_metric_label)
].copy()

if not df_b.empty:
    cap_min = df_b["Max samples per class"].min()
    cap_max = df_b["Max samples per class"].max()
    top_max_cap = df_b.loc[df_b[df_b["Max samples per class"] == cap_max]["Mean alignment score"].idxmax(), "Embedding"]
    top_min_cap = df_b.loc[df_b[df_b["Max samples per class"] == cap_min]["Mean alignment score"].idxmax(), "Embedding"]

    if top_max_cap == top_min_cap:
        balance_flag = "CONSISTENT"
        balance_note = f"Top model ({top_max_cap}) unchanged across class balance sweep."
    elif top_min_cap == primary_model:
        balance_flag = "FREQUENCY-DRIVEN"
        balance_note = f"Ranking flips from {top_max_cap} to {top_min_cap} at balanced classes."
    else:
        balance_flag = "INCONSISTENT"
        balance_note = f"Ranking flips from {top_max_cap} to {top_min_cap} at balanced classes."
else:
    balance_flag = "UNKNOWN"
    balance_note = f"No data for k={k_ref}, metric={primary_metric_label}."

print(f"\nStep 5 | Balance flag: {balance_flag}")
print(f"  {balance_note}")

### Step 6: Feature noise robustness 

`plot_knn_alignment_vs_feature_noise` progressively corrupts a fraction of labels while keeping the embedding geometry fixed. A score that degrades gracefully with noise indicates that the class structure is genuinely encoded in the embeddings, rather than being an artefact of a few perfectly labelled hubs.

In [ ]:
from emmaemb.visualization import plot_knn_alignment_vs_feature_noise

fig_noise, df_noise = plot_knn_alignment_vs_feature_noise(
    emma=emma,
    feature=feature,
    emb_spaces=models,
    k_values=k_values,
    metrics=distance_metrics,
    n_noise_steps=8,
    n_repeats=3,
    seed=42,
    show_random_baselines=True,
    return_data=True,
)
fig_noise.update_layout(height=900, width=750)
fig_noise.show()

# Step 6: noise robustness
metric_display = {"cosine": "Cosine distance", "cityblock": "Manhattan distance", "euclidean": "Euclidean distance"}
k_ref = k_values[len(k_values) // 2]
primary_metric_label = metric_display.get(distance_metrics[0], distance_metrics[0])

df_n = (
    df_noise[(df_noise["k"] == k_ref) & (df_noise["Metric"] == primary_metric_label)]
    .groupby(["Noise fraction", "Embedding"])["Mean alignment score"]
    .mean()
    .unstack("Embedding")
)

if not df_n.empty and len(df_n.columns) > 1:
    leader_at_zero = df_n.iloc[0].idxmax()
    crossings = df_n[df_n.idxmax(axis=1) != leader_at_zero]
    if crossings.empty or float(crossings.index[0]) > 0.3:
        noise_flag = "ROBUST"
        crossing_frac = float(crossings.index[0]) if not crossings.empty else None
        noise_note = (
            f"Rankings stable up to noise={crossing_frac:.2f}."
            if crossing_frac is not None
            else "Rankings stable across all noise levels."
        )
    else:
        noise_flag = "SENSITIVE"
        crossing_frac = float(crossings.index[0])
        noise_note = f"Rankings flip at noise={crossing_frac:.2f}, class structure is noise-sensitive."
else:
    noise_flag = "ROBUST"
    noise_note = "Single model or no data, noise robustness not applicable."

print(f"\nStep 6 | Noise flag: {noise_flag}")
print(f"  {noise_note}")

# Overall confidence assessment
df_ref = df_knn[df_knn["k"] == k_ref].groupby(["distance_metric", "Embedding"])["Fraction"].mean()
margins = []
for metric_key in distance_metrics:
    if metric_key in df_ref.index.get_level_values("distance_metric"):
        grp = df_ref.xs(metric_key, level="distance_metric").sort_values(ascending=False)
        if len(grp) >= 2:
            margins.append(float(grp.iloc[0] - grp.iloc[1]))
margin = float(np.mean(margins)) if margins else 0.0

all_robust = stability_flag == "STABLE" and balance_flag == "CONSISTENT" and noise_flag == "ROBUST"
any_unreliable = (
    stability_flag == "PARAMETER-SENSITIVE"
    or balance_flag == "INCONSISTENT"
    or noise_flag == "SENSITIVE"
)

print(f"\n=== Final Assessment ===")
print(f"  Stability:    {stability_flag}")
print(f"  Balance:      {balance_flag}")
print(f"  Noise:        {noise_flag}")
print(f"  Primary model: {primary_model}")

### A.2 Within/between distance distributions

`plot_within_between_distributions` visualises the overlap between within-class and between-class pairwise distances. A clean separation indicates the embedding geometry reflects the class structure well.

In [ ]:
from emmaemb.visualization import plot_within_between_distributions

fig_wb = plot_within_between_distributions(
    emma=emma,
    emb_space="ProtT5",
    metric="cityblock",
    feature="enzyme_class",
)
fig_wb.update_layout(height=500, width=700)
fig_wb.show()

### KNN class mixing matrix

The KNN class mixing matrix quantifies the mixing of classes of one feature within the KNN neighborhood of samples in a given embedding space. In this ecample we look at the class mixing for the enzyme classes.
Given a distance metric (e.g. cosine), the KNN are retrieved for each sample and the KNN class `get_class_mixing_in_neighborhood` counts how often different classes appear among its neighbors. The function returns a class mixing matrix, where each entry represents the number of times a class appears in the neighborhood of another class, along with the unique class labels. The heatmap can be visualised using the `plot_knn_class_mixing_matrix` function.

In [ ]:
# KNN CLASS MIXING MATRIX
from emmaemb.visualization import plot_knn_class_mixing_matrix

fig_class_mixing_matrix = plot_knn_class_mixing_matrix(
    emma,
    emb_space="ProtT5",
    feature="enzyme_class",
    k=100,
    metric="cosine",
)
fig_class_mixing_matrix.update_layout(height=600, width=600)
fig_class_mixing_matrix.show()

<a name="section-four"></a>
## IV. Pairwise space comparison

### Global comparison of pairwise distances

The `plot_pairwise_distance_comparison` function generates a scatter plot to compare pairwise distances between samples in two different embedding spaces. Using a specified distance metric (e.g. cosine), it shows the distances for the same set of samples across both embedding spaces.
Additionally it computes the Spearman correlation coefficient between the pairwise distances in the two selected embedding spaces.
The function allows customization of plot title, color, and scatter dot opacity, and optionally groups points based on a meta data feature, enabling insights into how different sample categories behave across embeddings.

In [ ]:
from emmaemb.visualization import plot_pairwise_distance_comparison

fig_pwd_comparison = plot_pairwise_distance_comparison(
    emma,
    emb_space_y="ProtT5",
    emb_space_x="ESMC",
    metric="cosine",
    group_by="species",
)
fig_pwd_comparison.update_layout(height=600, width=600)
fig_pwd_comparison.show()

### Local cross-space neighborhood similarity

The `plot_low_similarity_distribution` function visualizes the class distribution of samples with low neighborhood similarity between two embedding spaces. It computes neighborhood similarity scores based on a specified distance metric (default: euclidean) and identifies samples where similarity of the nearest neighbors of a data point falls below a given threshold. The function then compares the class distribution of these low-similarity samples to the overall dataset distribution using a scatter plot. This helps assess whether certain classes exhibit higher or lower structural consistency across embeddings, providing insights into differences in how embeddings capture relationships between samples.

In [ ]:
from emmaemb.visualization import plot_low_similarity_distribution

fig_low_similarity_class_distribution = plot_low_similarity_distribution(
    emma,
    emb_space_1="ProtT5",
    emb_space_2="ESMC",
    feature="enzyme_class",
    k=10,
    metric="cosine",
    similarity_threshold=0.3,
)
fig_low_similarity_class_distribution.update_layout(height=600, width=600)
fig_low_similarity_class_distribution.show()